In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

### Task 1: Build a Reusable `bootstrap_ci` Function

Write a function with the following signature:

```python
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.

    Parameters
    ----------
    data : array-like
        The sample to resample from.
    stat_func : callable
        The statistic to compute on each resample (default: np.mean).
    n_boot : int
        Number of bootstrap resamples.
    ci_level : float
        Confidence level (e.g. 0.95 for a 95 % CI).
    """
```

**Steps:**

1. Inside the function, use `np.random.choice` (with replacement) to draw `n_boot` resamples of the same size as `data`.
2. Compute `stat_func` on each resample and store the results in an array called `boot_stats`.
3. Use `np.percentile` to extract the lower and upper bounds based on `ci_level`. For a 95 % CI these are the 2.5th and 97.5th percentiles.
4. Return `(lower, upper)`.

**Validation:** Call your function on `np.arange(1, 101)` with the default mean. The 95 % CI should be roughly (40, 60). Verify this before moving on.

In [2]:
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.
    """
    boot_stats = []
    n = len(data)
    
    for _ in range(n_boot):
        # Step 1: Resample with replacement
        resample = np.random.choice(data, size=n, replace=True)
        # Step 2: Compute the statistic
        boot_stats.append(stat_func(resample))
    
    # Step 3: Calculate percentiles
    alpha = (1 - ci_level) / 2
    lower = np.percentile(boot_stats, alpha * 100)
    upper = np.percentile(boot_stats, (1 - alpha) * 100)
    
    return lower, upper

# Validation
test_data = np.arange(1, 101)
lower, upper = bootstrap_ci(test_data)
print(f"Validation 95% CI for mean of 1-100: ({lower:.2f}, {upper:.2f})")
# Expected: Roughly (40, 60)

Validation 95% CI for mean of 1-100: (44.80, 56.10)


### Task 2: Apply Bootstrap CIs to Real Data

Load or generate a dataset with at least 200 rows that includes:

- A **continuous numeric** column (e.g. income, test score, response time)
- A **binary / boolean** column (e.g. purchased yes/no, passed yes/no)

You may use any publicly available dataset or generate synthetic data with `np.random.normal` / `np.random.binomial`.

**Compute the following bootstrap CIs (95 %, 10 000 resamples each):**

| # | Statistic | Column | `stat_func` |
|---|-----------|--------|-------------|
| 1 | Mean | Continuous column | `np.mean` |
| 2 | Median | Continuous column | `np.median` |
| 3 | Proportion | Binary column | `np.mean` (on 0/1 values) |

Present the three intervals in a tidy summary table using `pd.DataFrame`.

In [3]:
# Generate synthetic data
n_samples = 250
df = pd.DataFrame({
    'spend': np.random.gamma(shape=2, scale=50, size=n_samples), # Skewed continuous data
    'is_premium': np.random.binomial(n=1, p=0.3, size=n_samples) # Binary data
})

# Compute Bootstrap CIs
ci_mean = bootstrap_ci(df['spend'], np.mean)
ci_median = bootstrap_ci(df['spend'], np.median)
ci_prop = bootstrap_ci(df['is_premium'], np.mean)

# Summary Table
summary_df = pd.DataFrame({
    'Statistic': ['Mean (Spend)', 'Median (Spend)', 'Proportion (Premium)'],
    'Lower Bound': [ci_mean[0], ci_median[0], ci_prop[0]],
    'Upper Bound': [ci_mean[1], ci_median[1], ci_prop[1]]
})

display(summary_df)

,Statistic,Lower Bound,Upper Bound
0,Mean (Spend),84.217188,100.956222
1,Median (Spend),64.544449,88.372625
2,Proportion (Premium),0.264000,0.380000


### Task 3: Compare Bootstrap CIs with Normal-Approximation CIs

For the **mean** and the **proportion** from Task 2, compute classical normal-approximation 95 % CIs:

- **Mean CI:** Use `st.t.interval` (or `st.norm.interval`) with the sample mean and standard error.
- **Proportion CI:** Use the Wald interval formula: p̂ ± z × √(p̂(1 − p̂) / n), where z = 1.96.

Create a comparison table with columns: `Statistic`, `Bootstrap Lower`, `Bootstrap Upper`, `Normal Lower`, `Normal Upper`.

**Answer in a Markdown cell:**

1. Are the two approaches giving similar intervals? Where do they diverge, if at all?
2. For which statistic (mean vs. proportion vs. median) is the bootstrap approach especially useful, and why?

In [4]:
# 1. Normal CI for Mean
mean_val = df['spend'].mean()
sem = st.sem(df['spend'])
norm_mean_ci = st.t.interval(0.95, len(df['spend'])-1, loc=mean_val, scale=sem)

# 2. Normal (Wald) CI for Proportion
p_hat = df['is_premium'].mean()
n = len(df['is_premium'])
z = 1.96
margin_error = z * np.sqrt((p_hat * (1 - p_hat)) / n)
norm_prop_ci = (p_hat - margin_error, p_hat + margin_error)

# Comparison Table
comparison_df = pd.DataFrame({
    'Statistic': ['Mean', 'Proportion'],
    'Bootstrap Lower': [ci_mean[0], ci_prop[0]],
    'Bootstrap Upper': [ci_mean[1], ci_prop[1]],
    'Normal Lower': [norm_mean_ci[0], norm_prop_ci[0]],
    'Normal Upper': [norm_mean_ci[1], norm_prop_ci[1]]
})

display(comparison_df)

,Statistic,Bootstrap Lower,Bootstrap Upper,Normal Lower,Normal Upper
0,Mean,84.217188,100.956222,83.887277,100.960056
1,Proportion,0.264000,0.380000,0.262175,0.377825


**1. Are the two approaches giving similar intervals? Where do they diverge?**
In many cases, the intervals are quite similar, especially for the **mean** if the sample size is large (due to the Central Limit Theorem). However, they often diverge when the underlying data is heavily skewed (like our 'spend' data). The Bootstrap interval is more "honest" about the data's actual distribution, whereas the Normal-approximation forces a symmetric interval that might include impossible values or miss the skew.

**2. For which statistic is the bootstrap approach especially useful, and why?**
The bootstrap is especially useful for the **Median**. Unlike the mean, there isn't a simple, universally reliable formula for the "Standard Error of the Median" that works for all distributions. Bootstrap handles this effortlessly by simply recalculating the median thousands of times, making no assumptions about the shape of the data.